# Mapping problems: Procrustes matching via SDP relaxation

Given two point clouds $P, Q \in \mathbb{R}^{d \times n}$, the Procrustes matching (PM) problem asks for a rotation $R \in O(d)$ and a permutation $X \in \Pi_n$ such that
$$ \min_{R \in O(d),\ X \in \Pi_n} \|RP - QX\|_F^2. $$
The problem is non-convex because rotations and permutations are both non-convex sets, and the permutation search is combinatorial.

A common baseline is iterative closest point (ICP): alternate between solving for $R$ (closed-form SVD) and solving for $X$ (linear assignment). ICP is easy to run but sensitive to initialization and can converge to local minima.

Maron, Dym, Kezurer, Kovalsky, and Lipman (2016) propose PM-SDP, an SDP relaxation that lifts the coupling between $R$ and $X$. For asymmetric isometric shapes, the relaxation is tight and recovers the global PM solution.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import scipy.io as sio
import scipy.sparse.linalg as spl
import matplotlib.pyplot as plt
import igl
from src.pmsdp_solve import pmsdp_solve
from src.pmsdp_interleaving import pmsdp_interleaving
from src.pmsdp_svd import pmsdp_svd

## Building point clouds from a real mesh

We use the SCAPE assets bundled with the paper code (`mesh000.off`, `mesh037.off`) and their pre-selected feature points.

Following the paper, we work in a functional-map embedding: evaluate Laplace-Beltrami eigenfunctions at vertices and drop the constant mode. Each selected feature then becomes a vector in $\mathbb{R}^{\text{prob\_dim}}$.

In this notebook we build a controlled test: sample feature points, embed them, and construct
$$ P = R^\star Q X^\star $$
with a random orthogonal matrix and a random permutation. The solver should recover both exactly.

In [ ]:
scape_dir = pathlib.Path('../external/PM-SDP/code/shapes/SCAPE/meshes')
V_mesh, F_mesh = igl.read_triangle_mesh(str(scape_dir / 'mesh000.off'))
print(f'mesh000.off: {V_mesh.shape[0]} vertices, {F_mesh.shape[0]} faces')

prob_dim = 8
L = -igl.cotmatrix(V_mesh, F_mesh)
M = igl.massmatrix(V_mesh, F_mesh, igl.MASSMATRIX_TYPE_VORONOI)
evals, evecs = spl.eigsh(L, k=prob_dim + 1, M=M, sigma=0.0, which='LM')
phi = evecs[:, 1:prob_dim + 1]

feature_points = sio.loadmat(
    str(scape_dir / 'mesh000.mat'),
    simplify_cells=True,
 )['mesh']['featurePoints'].astype(int) - 1
n = 12
idx = feature_points[:n]
print(f'using the first {n} of {len(feature_points)} pre-selected feature points')

Q = phi[idx, :].T

rng = np.random.default_rng(0)
real_R, _ = np.linalg.qr(rng.standard_normal((prob_dim, prob_dim)))
real_R *= np.sign(np.linalg.det(real_R))
perm = rng.permutation(n)
real_X = np.eye(n)[perm, :]
P = real_R.T @ Q @ real_X
print(f'true permutation: {perm.tolist()}')

## Naive ICP from random initializations

ICP alternates two convex subproblems: Procrustes update for $R$ and assignment update for $X$. The overall objective is still non-convex, so different starts can converge to different stationary points.

Below we run ICP from multiple random initial permutations and report the final residuals.

In [ ]:
n_starts = 10
icp_residuals = []

for s in range(n_starts):
    rng_start = np.random.default_rng(100 + s)
    perm_init = rng_start.permutation(n)
    X0 = np.eye(n)[perm_init, :]
    R0 = pmsdp_svd(P, Q @ X0)
    Xp, Rp, residual, n_iter = pmsdp_interleaving(X0, R0, P, Q, start_with='X')
    icp_residuals.append(residual)

print(f'ICP residuals over {n_starts} random starts:')
for residual in sorted(icp_residuals):
    print(f'{residual:.4f}')
print('Zero residual means global recovery.')

Most runs stop at non-zero residual, which indicates incorrect correspondences. Only starts close to the true permutation typically converge to the global solution.

## PM-SDP

We now solve the relaxed problem with `pmsdp_solve`, which builds the lifted PSD blocks, imposes doubly-stochastic and orthogonality constraints, and calls the SDP backend.

After solving, we project the relaxed solution to a permutation and a rotation using the same interleaving-style postprocess used in the original implementation.

In [ ]:
X_sdp, R_sdp = pmsdp_solve(P, Q, verbose=False)

print(f'||X_sdp - X*||_F  : {np.linalg.norm(X_sdp - real_X, "fro"):.2e}')
print(f'||R_sdp - R*||_F  : {np.linalg.norm(R_sdp - real_R, "fro"):.2e}')
print(f'final residual    : {np.linalg.norm(R_sdp @ P - Q @ X_sdp, "fro"):.2e}')

Both $X$ and $R$ are recovered to numerical precision. Unlike ICP, the solve does not rely on random initialization.

## Visualization

We display two SCAPE meshes side by side, mark source and matched target feature points, and connect matches with line segments. At convergence, each segment collapses as the mapped points coincide.

In [ ]:
import polyscope as ps

V_target, F_target = igl.read_triangle_mesh(str(scape_dir / 'mesh037.off'))

x = V_mesh[:, 0]
colour_field = (x - x.min()) / (x.max() - x.min())

shift = np.array([0.0, 0.0, 1.6 * (V_mesh[:, 2].max() - V_mesh[:, 2].min())])
V_target_shifted = V_target + shift

recovered = X_sdp.argmax(axis=0)
perm_inv = np.argsort(perm)
src_xyz = V_mesh[idx[perm_inv]]
tgt_xyz = V_target_shifted[idx[recovered]]

ps.init()
ps.remove_all_structures()
ps.set_ground_plane_mode('none')
ps.set_view_projection_mode('orthographic')

ps_src = ps.register_surface_mesh('mesh000 (source)', V_mesh, F_mesh, smooth_shade=True)
ps_src.add_scalar_quantity('colour', colour_field, cmap='turbo', enabled=True)

ps_tgt = ps.register_surface_mesh('mesh037 (target)', V_target_shifted, F_target, smooth_shade=True)
ps_tgt.add_scalar_quantity('colour', colour_field, cmap='turbo', enabled=True)

ps.register_point_cloud('source feature points', src_xyz).set_color((1.0, 1.0, 1.0))
ps.register_point_cloud('matched target points', tgt_xyz).set_color((1.0, 1.0, 1.0))

pts = np.vstack([src_xyz, tgt_xyz])
edges = np.stack([np.arange(n), n + np.arange(n)], axis=1)
ps.register_curve_network('correspondences', pts, edges).set_radius(0.002)
ps.show()

**▶ Your turn.** Replace `Q` with the embedding built from `mesh037.off` while keeping the same anatomical feature indices. This is the cross-shape setting from the paper.

You should observe that this simplified notebook may lose exact recovery because it omits two paper-side constraints: banding on $R$ (`utilizeRFlag` / `Rtol`) and the AGD-based mask on $X$ (`UtilizeXflag` / `permConstraint`). Extending the interleaving projection with the same banding logic is the natural next implementation step.

## References

[1] *Point matching via semidefinite programming*.
       Maron, H.; Dym, N.; Kezurer, I.; Kovalsky, S.; Lipman, Y.
       ACM Transactions on Graphics (SIGGRAPH Asia) 35 (6): 1 (2016).
       :DOI:`10.1145/2980179.2982402`